In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install transformers accelerate deepspeed peft datasets bitsandbytes
!pip install tensorboard ninja

# Stage 4 — Distributed Training
## Notebook 1: DeepSpeed, Accelerate, and FSDP

**Learning Objectives:**
- Understand DeepSpeed ZeRO stages (1/2/3) with memory calculation formulas
- Configure and launch multi-GPU training with HuggingFace Accelerate
- Apply FSDP and compare it to DeepSpeed
- Write production-grade distributed launch scripts for SLURM clusters

**Prerequisites:** Stages 1–3, familiarity with PyTorch training loops.

---
## 1. DeepSpeed ZeRO — Memory Breakdown

When training large models, GPU memory is consumed by four components:

| Component | Symbol | Size (7B model, fp16) |
|-----------|--------|-----------------------|
| Parameters | $\Psi$ | 14 GB |
| Gradients | $\nabla$ | 14 GB |
| Optimizer States | $\mathcal{O}$ | 84 GB (Adam in fp32) |
| Activations | $\mathcal{A}$ | 20–60 GB |

**Total without ZeRO: ~132–172 GB** -- far exceeding any single GPU.

### ZeRO Stage Formulae
Let $N$ = number of GPUs:

- **ZeRO-1** (Optimizer State Partitioning):  
  $\text{Mem}_{\text{GPU}} = \Psi + \nabla + \frac{\mathcal{O}}{N} + \mathcal{A}$

- **ZeRO-2** (Optimizer + Gradient Partitioning):  
  $\text{Mem}_{\text{GPU}} = \Psi + \frac{\nabla}{N} + \frac{\mathcal{O}}{N} + \mathcal{A}$

- **ZeRO-3** (Full Parameter + Gradient + Optimizer Sharding):  
  $\text{Mem}_{\text{GPU}} = \frac{\Psi}{N} + \frac{\nabla}{N} + \frac{\mathcal{O}}{N} + \mathcal{A}$

> **Worked Example (7B model, 8 GPUs):**  
> ZeRO-1: 14 + 14 + 84/8 + 40 = 78.5 GB/GPU  
> ZeRO-2: 14 + 14/8 + 84/8 + 40 = 66.3 GB/GPU  
> ZeRO-3: 14/8 + 14/8 + 84/8 + 40 = 54.0 GB/GPU

In [ ]:
# Memory calculator for any model size
def zero_memory(num_params_b, num_gpus, activation_gb=40.0,
                 bytes_per_param=2, adam_multiplier=12):
    """Estimate per-GPU memory for each ZeRO stage.
    Args:
        num_params_b: parameters in billions (e.g. 7.0 for 7B)
        num_gpus: number of GPUs
        activation_gb: estimated activation memory in GB
        bytes_per_param: 2 for fp16, 4 for fp32
        adam_multiplier: 12 for Adam (fp32 param + exp_avg + exp_avg_sq)
    """
    P = num_params_b * bytes_per_param
    G = num_params_b * bytes_per_param
    O = num_params_b * adam_multiplier
    A = activation_gb

    zero1 = P + G + O / num_gpus + A
    zero2 = P + G / num_gpus + O / num_gpus + A
    zero3 = P / num_gpus + G / num_gpus + O / num_gpus + A

    print(f"{'='*55}")
    print(f"Model: {num_params_b}B | GPUs: {num_gpus} | fp{'16' if bytes_per_param==2 else '32'}")
    print(f"Params={P:.1f}GB  Grads={G:.1f}GB  Optim={O:.1f}GB  Activ={A:.1f}GB")
    print(f"{'='*55}")
    print(f"  ZeRO-1 (Optim sharded):       {zero1:7.1f} GB/GPU")
    print(f"  ZeRO-2 (Optim+Grad sharded):   {zero2:7.1f} GB/GPU")
    print(f"  ZeRO-3 (Optim+Grad+Param):     {zero3:7.1f} GB/GPU")
    print(f"  No ZeRO (DP only):             {(P+G+O+A):7.1f} GB/GPU")

zero_memory(7.0, 8)
print()
zero_memory(13.0, 8)
print()
zero_memory(70.0, 16)

## 2. DeepSpeed Configuration JSON Files

DeepSpeed is configured via a JSON file passed to `deepspeed.initialize()`. Key configuration zones: `zero_optimization`, `optimizer`, `scheduler`, `fp16`/`bf16`, and `gradient_clipping`.

In [ ]:
import json

# ____ ZeRO-1 Config — Optimizer states sharded only ____
ZERO1_CONFIG = {
    "train_batch_size": 32,
    "train_micro_batch_size_per_gpu": 4,
    "gradient_accumulation_steps": 1,
    "fp16": {
        "enabled": True,
        "loss_scale": 0,
        "loss_scale_window": 1000,
        "hysteresis": 2,
        "min_loss_scale": 1
    },
    "zero_optimization": {
        "stage": 1,
        "reduce_bucket_size": 5e8,
        "allgather_bucket_size": 5e8,
        "overlap_comm": True,
        "contiguous_gradients": True
    },
    "optimizer": {
        "type": "AdamW",
        "params": {"lr": 2e-5, "betas": [0.9, 0.999],
                   "eps": 1e-8, "weight_decay": 0.01}
    },
    "scheduler": {
        "type": "WarmupDecayLR",
        "params": {"warmup_min_lr": 0, "warmup_max_lr": 2e-5,
                   "warmup_num_steps": 500, "total_num_steps": 10000}
    },
    "gradient_clipping": 1.0
}
with open("ds_config_zero1.json", "w") as f:
    json.dump(ZERO1_CONFIG, f, indent=2)
print("Wrote ds_config_zero1.json")

In [ ]:
# ____ ZeRO-2 Config — Optimizer + Gradient sharding ____
ZERO2_CONFIG = {
    "train_batch_size": 64,
    "train_micro_batch_size_per_gpu": 2,
    "gradient_accumulation_steps": 4,
    "bf16": {"enabled": True},
    "zero_optimization": {
        "stage": 2,
        "allgather_partitions": True,
        "allgather_bucket_size": 5e8,
        "overlap_comm": True,
        "reduce_scatter": True,
        "reduce_bucket_size": 5e8,
        "contiguous_gradients": True,
        "round_robin_gradients": True
    },
    "optimizer": {
        "type": "AdamW",
        "params": {"lr": 1e-4, "betas": [0.9, 0.95],
                   "eps": 1e-8, "weight_decay": 0.1}
    },
    "scheduler": {
        "type": "WarmupLR",
        "params": {"warmup_min_lr": 0, "warmup_max_lr": 1e-4,
                   "warmup_num_steps": 100}
    },
    "gradient_clipping": 1.0,
    "communication_data_type": "fp16"
}
with open("ds_config_zero2.json", "w") as f:
    json.dump(ZERO2_CONFIG, f, indent=2)
print("Wrote ds_config_zero2.json")

In [ ]:
# ____ ZeRO-3 Config — Full sharding + CPU offload ____
ZERO3_CONFIG = {
    "train_batch_size": 128,
    "train_micro_batch_size_per_gpu": 1,
    "gradient_accumulation_steps": 16,
    "bf16": {"enabled": True},
    "zero_optimization": {
        "stage": 3,
        "offload_optimizer": {"device": "cpu", "pin_memory": True},
        "offload_param": {"device": "cpu", "pin_memory": True},
        "overlap_comm": True,
        "contiguous_gradients": True,
        "sub_group_size": 1e9,
        "reduce_bucket_size": 5e7,
        "stage3_prefetch_bucket_size": 5e7,
        "stage3_param_persistence_threshold": 1e6,
        "stage3_max_live_parameters": 1e9,
        "stage3_max_reuse_distance": 1e9,
        "stage3_gather_16bit_weights_on_model_save": True
    },
    "optimizer": {
        "type": "AdamW",
        "params": {"lr": 5e-5, "betas": [0.9, 0.95],
                   "eps": 1e-8, "weight_decay": 0.1}
    },
    "scheduler": {
        "type": "WarmupLR",
        "params": {"warmup_min_lr": 0, "warmup_max_lr": 5e-5,
                   "warmup_num_steps": 50}
    },
    "gradient_clipping": 1.0
}
with open("ds_config_zero3.json", "w") as f:
    json.dump(ZERO3_CONFIG, f, indent=2)
print("Wrote ds_config_zero3.json")

## 3. Accelerate: Single-GPU to Multi-GPU

Accelerate abstracts device placement, mixed precision, and distributed launch behind a minimal API. The same script runs on 1 GPU, 8 GPUs, or TPUs with zero code changes.

In [ ]:
from accelerate import Accelerator
from transformers import AutoModelForCausalLM, AutoTokenizer, get_scheduler
from torch.utils.data import DataLoader, Dataset
import torch

# 1. Initialize accelerator — handles ALL device logic
accelerator = Accelerator(
    mixed_precision="bf16",
    gradient_accumulation_steps=4,
    log_with="tensorboard",
)

# 2. Load model — NO .to(device) needed
model = AutoModelForCausalLM.from_pretrained("gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# 3. Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

# 4. Dummy dataset
class DummyDataset(Dataset):
    def __len__(self): return 512
    def __getitem__(self, idx):
        text = f"Example {idx} " * 64
        return tokenizer(text, return_tensors="pt", truncation=True,
                        max_length=128, padding="max_length")

train_loader = DataLoader(DummyDataset(), batch_size=2, shuffle=True)
lr_scheduler = get_scheduler("cosine", optimizer=optimizer,
                             num_warmup_steps=50, num_training_steps=500)

# 5. THE magic line — prepares everything for distributed
model, optimizer, train_loader, lr_scheduler = accelerator.prepare(
    model, optimizer, train_loader, lr_scheduler
)

# 6. Training loop — identical to single-GPU code
model.train()
for epoch in range(2):
    for batch in train_loader:
        with accelerator.accumulate(model):
            outputs = model(**batch)
            loss = outputs.loss
            accelerator.backward(loss)    # NOT loss.backward()
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
    accelerator.log({"loss": loss.item(), "epoch": epoch})

# 7. Save — only main process writes
accelerator.wait_for_everyone()
unwrapped = accelerator.unwrap_model(model)
if accelerator.is_main_process:
    unwrapped.save_pretrained("./accelerate_checkpoint")

accelerator.print(f"Done on {accelerator.num_processes} processes!")

## 4. Multi-GPU LoRA Fine-Tuning with Accelerate + PEFT

LoRA reduces trainable parameters by ~99.5%, making multi-GPU training feasible on consumer hardware.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling
)
from datasets import Dataset
import torch

# 4-bit base model (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

MODEL_NAME = "microsoft/phi-2"  # 2.7B, fits any consumer GPU
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config,
    device_map="auto", trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# LoRA config
lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05, bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: trainable params ~0.5% of total

# Dataset
dummy_data = [{"text": f"### Instruction: Explain {i}\n### Response: Topic {i} is"}
              for i in range(500)]
dataset = Dataset.from_list(dummy_data)

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True,
                    max_length=256, padding="max_length")

tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=dataset.column_names)

# Trainer (handles DDP internally)
training_args = TrainingArguments(
    output_dir="./lora_multi_gpu_output",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=200,
    save_total_limit=3,
    learning_rate=2e-4,
    bf16=True,
    ddp_find_unused_parameters=False,  # critical for LoRA: prevents hangs
    report_to="none",
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

print("Trainer ready. Call trainer.train() to start.")
# trainer.train()  # uncomment to run

## 5. FSDP — Fully Sharded Data Parallel

FSDP is PyTorch's native ZeRO-3 equivalent. It shards parameters, gradients, and optimizer states across GPUs.

| Feature | DeepSpeed ZeRO-3 | FSDP |
|---------|-----------------|------|
| Origin | Microsoft | Meta (PyTorch) |
| Integration | `deepspeed.initialize()` | `torch.distributed.fsdp` |
| CPU Offload | Built-in | Manual via `CPUOffload` |
| Mixed Precision | Config JSON | `MixedPrecision` wrapper |
| Maturity | Battle-tested | Stabilizing in PyTorch 2.0+ |

**Choose FSDP** if you want pure PyTorch with no external dependencies.  
**Choose DeepSpeed** if you need ZeRO-Infinity (NVMe offload) or advanced scheduler features.

In [ ]:
# ____ FSDP Example (standalone script) ____
FSDP_SCRIPT = r'''
import torch
import torch.distributed as dist
from torch.distributed.fsdp import (
    FullyShardedDataParallel as FSDP,
    MixedPrecision, ShardingStrategy, CPUOffload,
    BackwardPrefetch, StateDictType, FullStateDictConfig
)
from torch.distributed.fsdp.wrap import transformer_auto_wrap_policy
from transformers import AutoModelForCausalLM, AutoTokenizer
from functools import partial

def setup_fsdp(model_name="gpt2"):
    """Configure FSDP for a transformer model."""
    from transformers.models.gpt2.modeling_gpt2 import GPT2Block

    auto_wrap_policy = partial(
        transformer_auto_wrap_policy,
        transformer_layer_cls={GPT2Block},
    )

    mixed_precision_policy = MixedPrecision(
        param_dtype=torch.bfloat16,
        reduce_dtype=torch.bfloat16,
        buffer_dtype=torch.bfloat16,
    )

    model = AutoModelForCausalLM.from_pretrained(model_name)

    fsdp_model = FSDP(
        model,
        auto_wrap_policy=auto_wrap_policy,
        mixed_precision=mixed_precision_policy,
        sharding_strategy=ShardingStrategy.FULL_SHARD,
        cpu_offload=CPUOffload(offload_params=False),
        backward_prefetch=BackwardPrefetch.BACKWARD_PRE,
        device_id=torch.cuda.current_device(),
        limit_all_gathers=True,
    )
    return fsdp_model

# Checkpoint save
def save_fsdp_checkpoint(fsdp_model, path):
    save_policy = FullStateDictConfig(offload_to_cpu=True, rank0_only=True)
    with FSDP.state_dict_type(fsdp_model, StateDictType.FULL_STATE_DICT, save_policy):
        state_dict = fsdp_model.state_dict()
        if dist.get_rank() == 0:
            torch.save(state_dict, f"{path}/pytorch_model.bin")

# Checkpoint load
def load_fsdp_checkpoint(fsdp_model, path):
    load_policy = FullStateDictConfig(offload_to_cpu=True, rank0_only=True)
    with FSDP.state_dict_type(fsdp_model, StateDictType.FULL_STATE_DICT, load_policy):
        state_dict = torch.load(f"{path}/pytorch_model.bin", map_location="cpu")
        fsdp_model.load_state_dict(state_dict)

print("FSDP setup complete. Launch: torchrun --nproc_per_node=8 fsdp_train.py")
'''
with open("fsdp_train.py", "w") as f:
    f.write(FSDP_SCRIPT)
print("Wrote fsdp_train.py")

## 6. Distributed Launch Commands

Three main launchers, each with single-node and multi-node syntax.

In [ ]:
# _____ Launch Command Reference _____
print("=" * 65)
print("1. torchrun (PyTorch native, recommended)")
print("=" * 65)
print("# Single node, 8 GPUs:")
print("torchrun --nproc_per_node=8 train.py")
print()
print("# Multi-node (2 nodes, 8 GPUs each):")
print("torchrun --nnodes=2 --nproc_per_node=8 \\")
print("    --rdzv_backend=c10d --rdzv_endpoint=10.0.0.1:29400 \\")
print("    --node_rank=0 train.py")

print()
print("=" * 65)
print("2. accelerate launch (HuggingFace, simplest)")
print("=" * 65)
print("# First run: accelerate config  (one-time setup)")
print("accelerate launch --num_processes=8 train.py")
print()
print("# Multi-node:")
print("accelerate launch --num_machines=2 --num_processes=16 \\")
print("    --machine_rank=0 --main_process_ip=10.0.0.1 \\")
print("    --main_process_port=29500 train.py")

print()
print("=" * 65)
print("3. deepspeed launcher")
print("=" * 65)
print("# Single node:")
print("deepspeed --num_gpus=8 train.py --deepspeed ds_config_zero2.json")
print()
print("# Multi-node:")
print("deepspeed --num_gpus=8 --num_nodes=2 \\")
print("    --master_addr=10.0.0.1 --master_port=29500 \\")
print("    train.py --deepspeed ds_config_zero3.json")

print()
print("Preferred order: torchrun > accelerate launch > deepspeed")

## 7. Checkpoint Save/Resume Logic

Long training runs (hours/days) must survive preemption and crashes. Always save: **model weights + optimizer state + scheduler state + training step** as an atomic checkpoint.

In [ ]:
import os, json, glob, shutil
import torch

class DistributedCheckpointManager:
    """Handles save/resume for DeepSpeed and standard distributed training."""

    def __init__(self, output_dir, save_every_n_steps=500, keep_last_n=3, use_deepspeed=False):
        self.output_dir = output_dir
        self.save_every_n_steps = save_every_n_steps
        self.keep_last_n = keep_last_n
        self.use_deepspeed = use_deepspeed
        os.makedirs(output_dir, exist_ok=True)

    def should_save(self, global_step):
        return global_step > 0 and global_step % self.save_every_n_steps == 0

    def save(self, model, optimizer, scheduler, global_step, scaler=None, engine=None):
        """Save checkpoint atomically."""
        ckpt_dir = os.path.join(self.output_dir, f"step_{global_step}")
        os.makedirs(ckpt_dir, exist_ok=True)

        if self.use_deepspeed and engine is not None:
            engine.save_checkpoint(ckpt_dir, tag=f"step_{global_step}")
        else:
            model_to_save = model.module if hasattr(model, "module") else model
            torch.save(model_to_save.state_dict(), os.path.join(ckpt_dir, "model.pt"))
            torch.save(optimizer.state_dict(), os.path.join(ckpt_dir, "optimizer.pt"))
            torch.save(scheduler.state_dict(), os.path.join(ckpt_dir, "scheduler.pt"))
            if scaler is not None:
                torch.save(scaler.state_dict(), os.path.join(ckpt_dir, "scaler.pt"))

        meta = {"global_step": global_step}
        with open(os.path.join(ckpt_dir, "meta.json"), "w") as f:
            json.dump(meta, f)

        if torch.distributed.is_initialized():
            print(f"[Rank {torch.distributed.get_rank()}] Saved -> {ckpt_dir}")
        self._cleanup_old()

    def _cleanup_old(self):
        """Keep only last N checkpoints."""
        checkpoints = sorted(
            [d for d in glob.glob(os.path.join(self.output_dir, "step_*")) if os.path.isdir(d)],
            key=lambda x: int(x.split("_")[-1])
        )
        for old in checkpoints[:-self.keep_last_n]:
            shutil.rmtree(old)
            print(f"Cleaned old checkpoint: {old}")

    def find_latest(self):
        """Find the most recent checkpoint for auto-resume."""
        checkpoints = sorted(
            [d for d in glob.glob(os.path.join(self.output_dir, "step_*")) if os.path.isdir(d)],
            key=lambda x: int(x.split("_")[-1])
        )
        return checkpoints[-1] if checkpoints else None

    def resume(self, model, optimizer, scheduler, engine=None):
        """Auto-resume from latest checkpoint. Returns starting global_step."""
        latest = self.find_latest()
        if latest is None:
            print("No checkpoint found — starting from scratch.")
            return 0

        print(f"Resuming from {latest}")
        if self.use_deepspeed and engine is not None:
            engine.load_checkpoint(latest)
        else:
            model.module.load_state_dict(torch.load(os.path.join(latest, "model.pt"), map_location="cuda"))
            optimizer.load_state_dict(torch.load(os.path.join(latest, "optimizer.pt"), map_location="cuda"))
            scheduler.load_state_dict(torch.load(os.path.join(latest, "scheduler.pt"), map_location="cuda"))

        with open(os.path.join(latest, "meta.json")) as f:
            meta = json.load(f)
        return meta["global_step"]

print("DistributedCheckpointManager defined.")
print("DeepSpeed checkpoint API:")
print("  engine.save_checkpoint(dir, tag=f'step_{step}')")
print("  _, client_state = engine.load_checkpoint(dir, tag=f'step_{step}')")

## 8. NCCL Debugging Environment Variables

NCCL (NVIDIA Collective Communications Library) is the backbone of GPU-to-GPU communication. When training hangs or crashes, these are your first diagnostic tools.

In [ ]:
import os

NCCL_ENV_GUIDE = {
    "NCCL_DEBUG": {
        "desc": "Set to INFO for general comm logging; TRACE for every collective.",
        "usage": "export NCCL_DEBUG=INFO"
    },
    "NCCL_DEBUG_FILE": {
        "desc": "Write NCCL debug output to a file instead of stderr.",
        "usage": "export NCCL_DEBUG_FILE=/tmp/nccl_rank_%h_%p.log"
    },
    "NCCL_SOCKET_IFNAME": {
        "desc": "Force NCCL to use a specific network interface.",
        "usage": "export NCCL_SOCKET_IFNAME=eth0"
    },
    "NCCL_IB_DISABLE": {
        "desc": "Set to 1 to disable InfiniBand (useful when IB is misconfigured).",
        "usage": "export NCCL_IB_DISABLE=1"
    },
    "NCCL_ASYNC_ERROR_HANDLING": {
        "desc": "Enable async error handling to prevent hangs on timeout — almost always set to 1.",
        "usage": "export NCCL_ASYNC_ERROR_HANDLING=1"
    },
    "NCCL_TIMEOUT": {
        "desc": "Timeout in seconds for collective operations (default 1800).",
        "usage": "export NCCL_TIMEOUT=3600"
    },
    "NCCL_P2P_DISABLE": {
        "desc": "Disable peer-to-peer GPU transfers (set 1 for debugging).",
        "usage": "export NCCL_P2P_DISABLE=1"
    },
    "NCCL_SHM_DISABLE": {
        "desc": "Disable shared memory transport (set 1 if /dev/shm is full).",
        "usage": "export NCCL_SHM_DISABLE=1"
    },
    "CUDA_VISIBLE_DEVICES": {
        "desc": "Restrict which GPUs are visible (not NCCL, but crucial for distributed training).",
        "usage": "export CUDA_VISIBLE_DEVICES=0,1,2,3"
    },
}

def set_nccl_debug(level="INFO", log_to_file=True):
    """Enable NCCL debug logging."""
    os.environ["NCCL_DEBUG"] = level
    os.environ["NCCL_ASYNC_ERROR_HANDLING"] = "1"
    if log_to_file:
        os.environ["NCCL_DEBUG_FILE"] = f"/tmp/nccl_rank_%h_%p.log"
    print(f"NCCL debug={level}, async_error_handling=1")

print("NCCL Environment Variables:")
for var, info in NCCL_ENV_GUIDE.items():
    print(f"  {var:32s} — {info['desc']}")
    print(f"  {'':32s}   {info['usage']}")

print()
print("Common recipes:")
print("  # Hang during init:")
print("  NCCL_DEBUG=INFO NCCL_DEBUG_SUBSYS=INIT torchrun train.py")
print("  # Timeout / watchdog firing:")
print("  NCCL_ASYNC_ERROR_HANDLING=1 TORCH_DISTRIBUTED_DEBUG=DETAIL torchrun train.py")
print("  # Slow performance (check topology):")
print("  nvidia-smi topo -m")

## 9. Communication Bottleneck Analysis

As you scale GPUs, communication overhead grows. The all-reduce time follows:  
$T_\text{allreduce} \approx 2 \cdot (n-1)/n \cdot \text{bytes} / \text{bandwidth}$

In [ ]:
def analyze_comm_overhead(
    num_gpus, model_params_b, batch_size, seq_len,
    hidden_dim=4096, nvlink_bw_gbps=600.0,
    network_bw_gbps=100.0, compute_tflops=312.0,
):
    """Estimate compute vs communication time for a training step."""
    grad_size_gb = model_params_b * 2  # fp16 gradients

    if num_gpus <= 8:  # single node, NVLink
        allreduce_ms = 2 * (num_gpus - 1) / num_gpus * grad_size_gb / nvlink_bw_gbps * 1000
    else:  # multi-node, network-limited
        allreduce_ms = 2 * (num_gpus - 1) / num_gpus * grad_size_gb / network_bw_gbps * 1000

    compute_flops = 6 * model_params_b * 1e9 * batch_size * seq_len
    compute_ms = compute_flops / (compute_tflops * 1e12) * 1000
    comm_ratio = allreduce_ms / max(compute_ms, 1e-6)

    print(f"Model: {model_params_b}B | GPUs: {num_gpus} | Batch: {batch_size} x Seq: {seq_len}")
    print(f"  Gradient all-reduce: {allreduce_ms:.2f} ms")
    print(f"  Compute (fwd+bwd):   {compute_ms:.2f} ms")
    print(f"  Comm/Compute ratio:  {comm_ratio:.4f}", end="")
    if comm_ratio > 0.1:
        print(" [WARNING] >10% overhead — consider grad accumulation or larger batch")
    else:
        print(" [OK]")
    print()

analyze_comm_overhead(8, 7.0, 4, 2048)
analyze_comm_overhead(64, 70.0, 1, 4096)
analyze_comm_overhead(256, 175.0, 1, 2048)

print("Note: With LoRA, gradient size drops to ~0.5% of full —")
print("making communication overhead negligible in most cases.")

## 10. SLURM Job Script for Cluster Scheduling

Production training runs on HPC clusters managed by SLURM.

In [ ]:
SLURM_SCRIPT = r'''#!/bin/bash
#SBATCH --job-name=llm_train
#SBATCH --nodes=4
#SBATCH --ntasks-per-node=8
#SBATCH --gpus-per-node=8
#SBATCH --cpus-per-task=8
#SBATCH --mem=0                          # all available memory
#SBATCH --time=72:00:00
#SBATCH --partition=gpu_a100
#SBATCH --output=logs/%x_%j.out
#SBATCH --error=logs/%x_%j.err
#SBATCH --exclusive

# Environment
source ~/.bashrc
conda activate llm_train

export CUDA_DEVICE_MAX_CONNECTIONS=1
export NCCL_DEBUG=WARN
export NCCL_ASYNC_ERROR_HANDLING=1
export NCCL_IB_TIMEOUT=22
export OMP_NUM_THREADS=$SLURM_CPUS_PER_TASK

# Master node discovery
MASTER_ADDR=$(scontrol show hostnames "$SLURM_JOB_NODELIST" | head -n 1)
MASTER_PORT=29500
echo "Master: $MASTER_ADDR:$MASTER_PORT"
echo "Total GPUs: $((SLURM_NNODES * SLURM_GPUS_PER_NODE))"

# Launch DeepSpeed
deepspeed \
    --num_nodes=$SLURM_NNODES \
    --num_gpus=$SLURM_GPUS_PER_NODE \
    --master_addr=$MASTER_ADDR \
    --master_port=$MASTER_PORT \
    train.py \
    --deepspeed ds_config_zero2.json \
    --model_name meta-llama/Llama-2-7b-hf \
    --output_dir ./output

# Alternative: torchrun (uncomment below)
# srun --ntasks=$((SLURM_NNODES * SLURM_GPUS_PER_NODE)) \
#     --gpus-per-task=1 \
#     torchrun --nnodes=$SLURM_NNODES \
#     --nproc_per_node=$SLURM_GPUS_PER_NODE \
#     --rdzv_backend=c10d \
#     --rdzv_endpoint=$MASTER_ADDR:$MASTER_PORT \
#     train.py

echo "Job completed at $(date)"
'''

with open("slurm_train.sh", "w") as f:
    f.write(SLURM_SCRIPT)
os.chmod("slurm_train.sh", 0o755)
print("Wrote slurm_train.sh")
print("Submit with: sbatch slurm_train.sh")
print()
print("Key SLURM commands:")
print("  sinfo                       # list partitions")
print("  squeue -u $USER             # your jobs")
print("  scancel <job_id>            # cancel job")
print("  srun --gres=gpu:4 --pty bash # interactive session")

## 11. Complete DeepSpeed Training Script

A full, launchable training script that works with `deepspeed --num_gpus=8 train.py`.

In [ ]:
DEEPSPEED_TRAIN_SCRIPT = r'''
"""train_deepspeed.py — Full DeepSpeed ZeRO-2 training script.

Usage:
    deepspeed --num_gpus=8 train_deepspeed.py \
        --deepspeed ds_config_zero2.json
"""
import argparse
import torch
import deepspeed
from transformers import AutoModelForCausalLM, AutoTokenizer
from torch.utils.data import DataLoader, Dataset

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model_name", type=str,
                        default="meta-llama/Llama-2-7b-hf")
    parser.add_argument("--output_dir", type=str, default="./output")
    parser.add_argument("--num_epochs", type=int, default=3)
    parser.add_argument("--local_rank", type=int, default=-1)
    parser = deepspeed.add_config_arguments(parser)
    return parser.parse_args()

def main():
    args = parse_args()

    model = AutoModelForCausalLM.from_pretrained(
        args.model_name, trust_remote_code=True)
    tokenizer = AutoTokenizer.from_pretrained(args.model_name)
    tokenizer.pad_token = tokenizer.eos_token

    class TextDataset(Dataset):
        def __len__(self): return 1024
        def __getitem__(self, idx):
            text = f"Training example {idx}: " + "text " * 32
            encoded = tokenizer(text, truncation=True, max_length=256,
                               padding="max_length", return_tensors="pt")
            return {"input_ids": encoded["input_ids"].squeeze(),
                    "attention_mask": encoded["attention_mask"].squeeze(),
                    "labels": encoded["input_ids"].squeeze()}

    dataloader = DataLoader(TextDataset(), batch_size=4,
                            shuffle=True, pin_memory=True)

    model_engine, optimizer, _, _ = deepspeed.initialize(
        args=args, model=model, model_parameters=model.parameters())

    global_step = 0
    for epoch in range(args.num_epochs):
        model_engine.train()
        for batch in dataloader:
            batch = {k: v.to(model_engine.device) for k, v in batch.items()}
            loss = model_engine(**batch).loss
            model_engine.backward(loss)
            model_engine.step()

            global_step += 1
            if global_step % 100 == 0:
                print(f"Step {global_step} | Loss: {loss.item():.4f}")
            if global_step % 500 == 0:
                model_engine.save_checkpoint(args.output_dir,
                                            tag=f"step_{global_step}")

    model_engine.save_checkpoint(args.output_dir, tag="final")
    print(f"Training complete. Output: {args.output_dir}")

if __name__ == "__main__":
    main()
'''

with open("train_deepspeed.py", "w") as f:
    f.write(DEEPSPEED_TRAIN_SCRIPT)
print("Wrote train_deepspeed.py")

---
## 12. Exercise: Write a ZeRO-2 Training Config & Script

**Task:** Build a complete DeepSpeed ZeRO-2 configuration and training script for fine-tuning `microsoft/phi-2` (2.7B params) on 4 GPUs.

**Requirements:**
1. bf16 mixed precision
2. Effective batch size = 32 (micro_batch=2, grad_accum=4, 4 GPUs)
3. AdamW optimizer: lr=5e-5, weight_decay=0.1
4. Cosine LR scheduler with 10% warmup
5. Gradient clipping at 1.0
6. Save checkpoint every 1000 steps, keep last 5

Fill in the blanks (`___`) below.

In [ ]:
# ____ Exercise: Fill in the ZeRO-2 config ____
EXERCISE_CONFIG = {
    "train_batch_size": ___,                  # effective batch = 32
    "train_micro_batch_size_per_gpu": ___,    # per-GPU micro batch = 2
    "gradient_accumulation_steps": ___,       # 4 GPUs x 2 x accum = 32 => accum = 4
    "bf16": {"enabled": ___},                 # True for Ampere/Ada GPUs
    "zero_optimization": {
        "stage": ___,                          # ZeRO-2
        "allgather_partitions": True,
        "allgather_bucket_size": 2e8,
        "overlap_comm": True,
        "reduce_scatter": True,
        "reduce_bucket_size": 2e8,
        "contiguous_gradients": True,
        "round_robin_gradients": True
    },
    "optimizer": {
        "type": ___,                            # AdamW
        "params": {
            "lr": ___,                          # 5e-5
            "betas": [0.9, 0.999],
            "eps": 1e-8,
            "weight_decay": ___                 # 0.1
        }
    },
    "scheduler": {
        "type": "WarmupDecayLR",
        "params": {
            "warmup_min_lr": 0,
            "warmup_max_lr": 5e-5,
            "warmup_num_steps": ___,            # 10% of total steps (100 of 1000)
            "total_num_steps": 1000
        }
    },
    "gradient_clipping": 1.0
}
print("Replace ___ with correct values.")

In [ ]:
# ____ Solution ____
SOLUTION = {
    "train_batch_size": 32,
    "train_micro_batch_size_per_gpu": 2,
    "gradient_accumulation_steps": 4,
    "bf16": {"enabled": True},
    "zero_optimization": {
        "stage": 2,
        "allgather_partitions": True,
        "allgather_bucket_size": 2e8,
        "overlap_comm": True,
        "reduce_scatter": True,
        "reduce_bucket_size": 2e8,
        "contiguous_gradients": True,
        "round_robin_gradients": True
    },
    "optimizer": {
        "type": "AdamW",
        "params": {
            "lr": 5e-5,
            "betas": [0.9, 0.999],
            "eps": 1e-8,
            "weight_decay": 0.1
        }
    },
    "scheduler": {
        "type": "WarmupDecayLR",
        "params": {
            "warmup_min_lr": 0,
            "warmup_max_lr": 5e-5,
            "warmup_num_steps": 100,
            "total_num_steps": 1000
        }
    },
    "gradient_clipping": 1.0
}

with open("ds_config_phi2_zero2.json", "w") as f:
    json.dump(SOLUTION, f, indent=2)

print("Solution written to ds_config_phi2_zero2.json")
print(f"Memory estimate:")
zero_memory(2.7, 4, activation_gb=20.0)
print()
print("Launch: deepspeed --num_gpus=4 train_phi2.py --deepspeed ds_config_phi2_zero2.json")

---
## 13. Summary

### Key Takeaways

| Concept | What You Learned |
|---------|-----------------|
| **ZeRO Stages** | ZeRO-1 shards optimizer; ZeRO-2 adds gradient sharding; ZeRO-3 adds parameter sharding |
| **Memory Formula** | Per-GPU mem = (sharded components)/N + replicated components + activations |
| **DeepSpeed Config** | JSON-driven: optimizer, scheduler, fp16/bf16, ZeRO stage, offloading |
| **Accelerate** | Minimal API: `accelerator.prepare()` handles DDP, mixed precision, device placement |
| **LoRA + Multi-GPU** | LoRA shrinks gradients to ~0.5%, making DDP communication negligible |
| **FSDP** | PyTorch-native ZeRO-3 equivalent; use `transformer_auto_wrap_policy` |
| **Launch Commands** | `torchrun` (recommended), `accelerate launch`, `deepspeed` |
| **Checkpointing** | Atomic save with model + optimizer + scheduler state; auto-resume from latest |
| **NCCL Debug** | `NCCL_DEBUG=INFO`, `NCCL_ASYNC_ERROR_HANDLING=1`, interface selection |
| **Comm Bottleneck** | Compute comm-to-compute ratio; aim for <10% overhead |
| **SLURM** | `#SBATCH` directives, `scontrol show hostnames` for master discovery |

### Next Step
Proceed to **Notebook 2** (`02_troubleshooting.ipynb`) for systematic diagnosis of training and inference failures.